# Import packages and configure reproducible sampling

In [1]:
import json
import hashlib
import copy
import io
import logging
import os
import random
import sys
import pandas as pd
import networkx as nx
import openai
from tqdm.auto import tqdm
from pathlib import Path
from contextlib import redirect_stdout

# Load credentials before importing TinyTroupe; its embedding client is created at import time.
env_path = Path("openai.env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            value = value.strip()
            if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '"'}:
                value = value[1:-1]
            os.environ[key.strip()] = value

# TinyTroupe prints its full configuration during import; keep notebook output clean.
with redirect_stdout(io.StringIO()):
    import tinytroupe
    from tinytroupe import config_manager
    from tinytroupe.agent import TinyPerson
    from tinytroupe.environment import TinyWorld

In [ ]:
RANDOM_SEED = 1
USE_CACHE = True  # Reuse valid population, name, and policy-response caches.
NETWORK_PRESET = "local_30"  # intimate_10, local_30, metro_75, regional_150, national_300
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "Add your key to openai.env as OPENAI_API_KEY=... and rerun this cell."
    )

# Short experiments do not need costly semantic-memory consolidation.
config_manager.update("enable_memory_consolidation", False)
config_manager.update("enable_continuous_contextual_semantic_memory_retrieval", False)
TinyPerson.MAX_EPISODE_LENGTH = 1000
TinyPerson.communication_display = False
TinyWorld.communication_display = False
logging.getLogger("tinytroupe").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.WARNING)

# Prepare a sample from the general Canadian Population

In [3]:
demographic_source_path = Path("data/canada_population.json")
piaac_source_path = Path("data/canada_piaac.json")
cache_path = Path("cache/canada_census_population.json")
cache_path.parent.mkdir(parents=True, exist_ok=True)

population_size = 300
sample_size = 300  # Generate the maximum sample once; the visualization can show smaller nested samples.

if sample_size > population_size:
    raise ValueError("sample_size cannot exceed population_size")

demographic_source_text = demographic_source_path.read_text(encoding="utf-8")
piaac_source_text = piaac_source_path.read_text(encoding="utf-8")
source_hash = hashlib.sha256(
    (demographic_source_text + piaac_source_text).encode()
).hexdigest()
demographic_source = json.loads(demographic_source_text)
piaac_source = json.loads(piaac_source_text)
demographic_dimensions = demographic_source["dimensions"]
piaac_dimensions = piaac_source["dimensions"]

def weighted_choice(rng, distribution):
    labels = list(distribution)
    weights = [distribution[label] for label in labels]
    return rng.choices(labels, weights=weights, k=1)[0]

demographic_distributions = {
    name: specification["categories"]
    for name, specification in demographic_dimensions.items()
}
piaac_distributions = {
    name: specification["categories"]
    for name, specification in piaac_dimensions.items()
}

proficiency_scores = {
    "Below Level 1": 0.05, "Level 1": 0.20, "Level 2": 0.40,
    "Level 3": 0.65, "Level 4": 0.85, "Level 5": 1.00,
}
category_scores = {
    "Category 1 - lowest": 0.10, "Category 2": 0.30, "Category 3": 0.50,
    "Category 4": 0.70, "Category 5 - highest": 0.90,
}
curiosity_scores = {
    "Lower - below -0.5 SD": 0.20, "Middle - -0.5 to 0.5 SD": 0.50,
    "Higher - above 0.5 SD": 0.80,
}
education_scores = {
    "Below high school completion": 0.20,
    "High school or other non-university credential": 0.55,
    "College or university credential": 0.85,
}

def ai_literacy_traits(profile, rng):
    literacy = proficiency_scores[profile["literacy_proficiency"]]
    numeracy = proficiency_scores[profile["numeracy_proficiency"]]
    problem_solving = proficiency_scores[profile["adaptive_problem_solving_proficiency"]]
    technology = (
        category_scores[profile["ict_use_at_home"]]
        + category_scores[profile["ict_use_at_work"]]
    ) / 2
    curiosity = curiosity_scores[profile["curiosity"]]
    learning = (
        0.45 * curiosity
        + 0.35 * category_scores[profile["learning_at_work"]]
        + 0.20 * (profile["recent_nonformal_education"] == "Participated")
    )
    education = education_scores[profile["education_attainment"]]

    def estimate(weighted_foundation, residual_sd=0.14):
        score = min(1, max(0, weighted_foundation + rng.gauss(0, residual_sd)))
        label = "Low" if score < 0.35 else "Moderate" if score < 0.65 else "High"
        return f"{label} ({round(score * 100)}/100; synthetic estimate)"

    return {
        "functional_ai_knowledge": estimate(
            0.32 * literacy + 0.18 * problem_solving + 0.20 * technology
            + 0.15 * education + 0.15 * learning
        ),
        "applied_ai_use_competence": estimate(
            0.10 * literacy + 0.15 * problem_solving + 0.40 * technology
            + 0.20 * learning + 0.15 * education
        ),
        "critical_ai_evaluation_capacity": estimate(
            0.40 * literacy + 0.25 * problem_solving + 0.15 * numeracy
            + 0.10 * technology + 0.10 * curiosity
        ),
        # PIAAC has no direct ethics measure, so this estimate has greater residual variance.
        "ethical_ai_understanding": estimate(
            0.25 * literacy + 0.20 * education + 0.15 * curiosity
            + 0.10 * learning + 0.30 * rng.random(),
            residual_sd=0.18,
        ),
    }

algorithm_version = 4
if USE_CACHE and cache_path.exists():
    cache = json.loads(cache_path.read_text(encoding="utf-8"))
else:
    cache = {}

cache_is_valid = (
    cache.get("source_hash") == source_hash
    and cache.get("algorithm_version") == algorithm_version
    and len(cache.get("profiles", [])) >= population_size
)

if cache_is_valid:
    demographic_population = cache["profiles"]
    print(f"Loaded {len(demographic_population)} cached demographic and PIAAC profiles.")
else:
    demographic_rng = random.Random(RANDOM_SEED)
    piaac_rng = random.Random(RANDOM_SEED + 10)
    ai_trait_rng = random.Random(RANDOM_SEED + 20)
    demographic_population = []

    for index in range(population_size):
        profile = {
            "profile_id": f"CA-SYN-{index + 1:06d}",
            "country": "Canada",
            **{
                name: weighted_choice(demographic_rng, distribution)
                for name, distribution in demographic_distributions.items()
            },
            **{
                name: weighted_choice(piaac_rng, distribution)
                for name, distribution in piaac_distributions.items()
            },
        }
        profile.update(ai_literacy_traits(profile, ai_trait_rng))
        demographic_population.append(profile)

    cache = {
        "source_hash": source_hash,
        "algorithm_version": algorithm_version,
        "random_seed": RANDOM_SEED,
        "profiles": demographic_population,
    }
    cache_path.write_text(
        json.dumps(cache, indent=2),
        encoding="utf-8",
    )

sampling_pool = demographic_population[:population_size]
sample_rng = random.Random(RANDOM_SEED + 1)
demographic_sample = sample_rng.sample(sampling_pool, k=sample_size)
pd.DataFrame(demographic_sample)

Loaded 1000 cached demographic and PIAAC profiles.


,profile_id,country,race_ethnicity,religious_affiliation,education_attainment,mother_tongue,urbanicity,household_composition,immigration_status,literacy_proficiency,...,self_assessed_performance,ict_use_at_home,ict_use_at_work,curiosity,learning_at_work,recent_nonformal_education,functional_ai_knowledge,applied_ai_use_competence,critical_ai_evaluation_capacity,ethical_ai_understanding
0,CA-SYN-000029,Canada,White,No religion or irreligion,College or university credential,English,Urban,Single-person household,Non-immigrant,Level 2,...,41-60%,Category 2,Category 1 - lowest,Middle - -0.5 to 0.5 SD,Category 3,Participated,Low (21/100; synthetic estimate),High (66/100; synthetic estimate),Moderate (46/100; synthetic estimate),Moderate (38/100; synthetic estimate)
1,CA-SYN-000047,Canada,White,Christianity,College or university credential,English,Urban,Single-person household,Non-immigrant,Level 4,...,81-100%,Category 3,Category 1 - lowest,Higher - above 0.5 SD,Category 3,Participated,Moderate (48/100; synthetic estimate),Moderate (37/100; synthetic estimate),Moderate (44/100; synthetic estimate),Moderate (48/100; synthetic estimate)
2,CA-SYN-000044,Canada,White,Christianity,College or university credential,Other,Urban,Single-person household,Non-immigrant,Level 3,...,41-60%,Category 2,Category 4,Middle - -0.5 to 0.5 SD,Category 3,Did not participate,Moderate (49/100; synthetic estimate),Low (28/100; synthetic estimate),Moderate (46/100; synthetic estimate),Low (32/100; synthetic estimate)
3,CA-SYN-000185,Canada,White,No religion or irreligion,College or university credential,English,Urban,Single-person household,Non-immigrant,Level 3,...,81-100%,Category 2,Category 5 - highest,Middle - -0.5 to 0.5 SD,Category 2,Participated,Moderate (56/100; synthetic estimate),High (69/100; synthetic estimate),Moderate (61/100; synthetic estimate),High (82/100; synthetic estimate)
4,CA-SYN-000087,Canada,White,Islam,High school or other non-university credential,English,Urban,Single-person household,Non-immigrant,Level 1,...,61-80%,Category 5 - highest,Category 3,Lower - below -0.5 SD,Category 5 - highest,Participated,Moderate (59/100; synthetic estimate),Moderate (56/100; synthetic estimate),Moderate (36/100; synthetic estimate),Moderate (46/100; synthetic estimate)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,CA-SYN-000277,Canada,White,No religion or irreligion,High school or other non-university credential,English,Urban,Single-person household,Non-immigrant,Level 4,...,81-100%,Category 4,Category 2,Middle - -0.5 to 0.5 SD,Category 1 - lowest,Did not participate,High (71/100; synthetic estimate),Moderate (45/100; synthetic estimate),High (74/100; synthetic estimate),High (82/100; synthetic estimate)
296,CA-SYN-000203,Canada,White,No religion or irreligion,College or university credential,English,Rural,Couple with or without children,Immigrant,Level 1,...,81-100%,Category 3,Category 5 - highest,Middle - -0.5 to 0.5 SD,Category 5 - highest,Did not participate,Low (33/100; synthetic estimate),High (66/100; synthetic estimate),Low (24/100; synthetic estimate),Low (6/100; synthetic estimate)
297,CA-SYN-000154,Canada,Filipino,No religion or irreligion,College or university credential,French,Urban,Single-parent household,Non-immigrant,Level 3,...,61-80%,Category 3,Category 2,Higher - above 0.5 SD,Category 5 - highest,Did not participate,Moderate (55/100; synthetic estimate),Moderate (45/100; synthetic estimate),High (67/100; synthetic estimate),Low (34/100; synthetic estimate)
298,CA-SYN-000236,Canada,White,Christianity,Below high school completion,Other,Rural,Couple with or without children,Non-immigrant,Level 1,...,81-100%,Category 3,Category 5 - highest,Lower - below -0.5 SD,Category 4,Did not participate,Low (26/100; synthetic estimate),Moderate (41/100; synthetic estimate),Moderate (39/100; synthetic estimate),Low (18/100; synthetic estimate)


In [4]:
names_cache_path = Path("cache/canada_profile_names.json")
if USE_CACHE and names_cache_path.exists():
    names_by_profile = json.loads(names_cache_path.read_text(encoding="utf-8"))
else:
    names_by_profile = {}

naming_profiles = {
    profile["profile_id"]: {
        key: value
        for key, value in profile.items()
        if key in {"profile_id", "country", *demographic_dimensions}
    }
    for profile in demographic_sample
}
profile_keys = {
    profile_id: hashlib.sha256(
        json.dumps(profile, sort_keys=True).encode()
    ).hexdigest()
    for profile_id, profile in naming_profiles.items()
}
missing_profiles = [
    naming_profiles[profile["profile_id"]]
    for profile in demographic_sample
    if profile_keys[profile["profile_id"]] not in names_by_profile
]

if missing_profiles:
    prompt = f"""
Assign one unique, plausible full name to each synthetic Canadian profile below.
Return only a JSON object with a 'names' object mapping each profile_id to its name.
Do not add titles, explanations, or extra IDs. Avoid caricatures and stereotypes.
Profiles: {json.dumps(missing_profiles, indent=2)}
"""
    client = openai.OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=config_manager.get("model"),
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    generated_names = json.loads(response.choices[0].message.content)["names"]

    expected_ids = {profile["profile_id"] for profile in missing_profiles}
    if set(generated_names) != expected_ids:
        raise RuntimeError("The generated-name response did not match the requested profiles.")
    if len(set(generated_names.values())) != len(generated_names):
        raise RuntimeError("The generated names were not unique.")

    for profile_id, name in generated_names.items():
        names_by_profile[profile_keys[profile_id]] = name
    names_cache_path.write_text(
        json.dumps(names_by_profile, indent=2),
        encoding="utf-8",
    )

TinyPerson.clear_agents()
tiny_people = []
for profile in demographic_sample:
    profile_key = profile_keys[profile["profile_id"]]
    person_name = names_by_profile[profile_key]
    agent = TinyPerson.get_agent_by_name(person_name)
    if agent is None:
        agent = TinyPerson(person_name)

    agent.define("nationality", "Canadian")
    agent.define("country_of_residence", "Canada")

    for field, value in profile.items():
        if field not in {"profile_id", "country"}:
            agent.define(field, value)

    tiny_people.append(agent)

[(person.name, person._persona) for person in tiny_people]

[('Olivia Scott',
  {'name': 'Olivia Scott',
   'age': None,
   'nationality': 'Canadian',
   'country_of_residence': 'Canada',
   'occupation': None,
   'race_ethnicity': 'White',
   'religious_affiliation': 'No religion or irreligion',
   'education_attainment': 'College or university credential',
   'mother_tongue': 'English',
   'urbanicity': 'Urban',
   'household_composition': 'Single-person household',
   'immigration_status': 'Non-immigrant',
   'literacy_proficiency': 'Level 2',
   'numeracy_proficiency': 'Below Level 1',
   'adaptive_problem_solving_proficiency': 'Level 3',
   'self_assessed_performance': '41-60%',
   'ict_use_at_home': 'Category 2',
   'ict_use_at_work': 'Category 1 - lowest',
   'curiosity': 'Middle - -0.5 to 0.5 SD',
   'learning_at_work': 'Category 3',
   'recent_nonformal_education': 'Participated',
   'functional_ai_knowledge': 'Low (21/100; synthetic estimate)',
   'applied_ai_use_competence': 'High (66/100; synthetic estimate)',
   'critical_ai_evalua

# Policy announcement simulation

Start with one policy to verify the workflow and API cost. Replace `selected_policy_ids` with `list(policy_data["announcements"])` to run all policies. Each selected policy requires one response from every agent.

In [5]:
policy_data = json.loads(
    Path("data/policy_announcements.json").read_text(encoding="utf-8")
)
strategic_landscape = json.loads(
    Path("data/canada_ai_strategic_landscape.json").read_text(encoding="utf-8")
)
landscape_for_simulation = {
    **strategic_landscape,
    "challenges": [
        {"name": challenge["name"], "context": challenge["context"]}
        for challenge in strategic_landscape["challenges"]
    ],
}

selected_policy_ids = ["ai_literacy"]
# selected_policy_ids = list(policy_data["announcements"])

unknown_policy_ids = set(selected_policy_ids) - set(policy_data["announcements"])
if unknown_policy_ids:
    raise KeyError(f"Unknown policy IDs: {sorted(unknown_policy_ids)}")

pd.DataFrame(
    [
        {"policy_id": policy_id, **policy_data["announcements"][policy_id]}
        for policy_id in selected_policy_ids
    ]
)

,policy_id,title,text
0,ai_literacy,Free introductory AI education,The Government of Canada announces plans for a...


In [6]:
class OneTalkWorld(TinyWorld):
    """Deliver at most one correctly targeted TALK action per agent and round."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.expected_targets = {}

    def _handle_actions(self, source, actions):
        expected = self.expected_targets.get(source.name)
        filtered_actions = []
        talk_kept = False
        for action in actions:
            if action.get("type") != "TALK":
                filtered_actions.append(action)
            elif not talk_kept and (action.get("target") or "") == expected:
                filtered_actions.append(action)
                talk_kept = True
        return super()._handle_actions(source, filtered_actions)

NETWORK_PRESETS = {
    "intimate_10": {
        "label": "Intimate group", "size": 10, "community_fields": [],
        "local_offsets": 3, "base_affinity": 5.0, "language_weight": 0.0,
        "urban_weight": 0.0, "education_weight": 0.0, "degree_base": 5,
        "degree_cap": 9, "pareto_shape": 3.0, "strong_tie_probability": 0.90,
    },
    "local_30": {
        "label": "Extended local community", "size": 30,
        "community_fields": ["urbanicity"], "local_offsets": 2,
        "base_affinity": 2.0, "language_weight": 2.0, "urban_weight": 2.0,
        "education_weight": 0.8, "degree_base": 4, "degree_cap": 12,
        "pareto_shape": 2.8, "strong_tie_probability": 0.82,
    },
    "metro_75": {
        "label": "Metropolitan-like community", "size": 75,
        "community_fields": ["mother_tongue"], "local_offsets": 2,
        "base_affinity": 2.5, "language_weight": 1.8, "urban_weight": 1.0,
        "education_weight": 0.6, "degree_base": 4, "degree_cap": 15,
        "pareto_shape": 2.5, "strong_tie_probability": 0.76,
    },
    "regional_150": {
        "label": "Regional or provincial public", "size": 150,
        "community_fields": ["urbanicity", "mother_tongue"], "local_offsets": 2,
        "base_affinity": 1.3, "language_weight": 2.5, "urban_weight": 1.5,
        "education_weight": 0.8, "degree_base": 4, "degree_cap": 18,
        "pareto_shape": 2.3, "strong_tie_probability": 0.72,
    },
    "national_300": {
        "label": "Canada-wide clustered public", "size": 300,
        "community_fields": ["urbanicity", "mother_tongue"], "local_offsets": 2,
        "base_affinity": 1.0, "language_weight": 3.0, "urban_weight": 2.0,
        "education_weight": 1.0, "degree_base": 4, "degree_cap": 18,
        "pareto_shape": 2.2, "strong_tie_probability": 0.68,
    },
}
if NETWORK_PRESET not in NETWORK_PRESETS:
    raise KeyError(f"Unknown NETWORK_PRESET: {NETWORK_PRESET}")
network_settings = NETWORK_PRESETS[NETWORK_PRESET]
unique_agent_profiles = []
seen_agent_names = set()
for person, profile in zip(tiny_people, demographic_sample):
    if person.name not in seen_agent_names:
        unique_agent_profiles.append((person, profile))
        seen_agent_names.add(person.name)
    if len(unique_agent_profiles) == network_settings["size"]:
        break
if len(unique_agent_profiles) < network_settings["size"]:
    raise RuntimeError(
        f"Preset requires {network_settings['size']} unique agents, but only "
        f"{len(unique_agent_profiles)} are available. Regenerate the name cache."
    )
active_people = [person for person, _ in unique_agent_profiles]
active_profiles = [profile for _, profile in unique_agent_profiles]

TinyWorld.clear_environments()
world = OneTalkWorld(
    "Canadian AI policy announcements",
    active_people,
    broadcast_if_no_target=False,
)
minimum_known_challenges = 2
maximum_known_challenges = 5
awareness_rng = random.Random(RANDOM_SEED + 30)
agent_strategic_awareness = {}

# Information is unevenly distributed: each agent knows only a reproducible random subset.
for agent in active_people:
    number_known = awareness_rng.randint(
        minimum_known_challenges, maximum_known_challenges
    )
    known_challenges = awareness_rng.sample(
        strategic_landscape["challenges"], k=number_known
    )
    agent_strategic_awareness[agent.name] = [
        challenge["name"] for challenge in known_challenges
    ]
    personal_context = (
        f"Background information you happen to know about {strategic_landscape['title']}:\n"
        + "\n".join(
            f"- {challenge['name']}: {challenge['context']}"
            for challenge in known_challenges
        )
        + f"\n\nInterpretation boundary: {strategic_landscape['interpretation_rule']}"
        + "\nUse this only as background when interpreting later announcements. "
          "Do not assume you know the omitted challenges, and do not recite this list. "
          "Let relevant background influence what you notice, prioritize, question, or worry about. "
          "Do not force it into every response or mention it merely to demonstrate knowledge."
    )
    agent.listen(personal_context, source=world, communication_display=False)
discussion_rounds = 3
response_word_limit = 40
network_rng = random.Random(RANDOM_SEED + 2)
profile_by_name = {
    person.name: profile for person, profile in zip(active_people, active_profiles)
}
agent_by_name = {person.name: person for person in active_people}
social_graph = nx.Graph()
social_graph.add_nodes_from(agent_by_name)

def community_of(name):
    profile = profile_by_name[name]
    fields = network_settings["community_fields"]
    return tuple(profile[field] for field in fields) if fields else ("all",)

def connection_affinity(left, right):
    a, b = profile_by_name[left], profile_by_name[right]
    return (
        network_settings["base_affinity"]
        + network_settings["language_weight"] * (a["mother_tongue"] == b["mother_tongue"])
        + network_settings["urban_weight"] * (a["urbanicity"] == b["urbanicity"])
        + network_settings["education_weight"] * (a["education_attainment"] == b["education_attainment"])
    )

communities = {}
for name in agent_by_name:
    communities.setdefault(community_of(name), []).append(name)
for members in communities.values():
    network_rng.shuffle(members)

# Dense local backbones create clustering inside demographic communities.
for members in communities.values():
    if len(members) > 1:
        for index, name in enumerate(members):
            for offset in range(1, network_settings["local_offsets"] + 1):
                if len(members) > offset:
                    social_graph.add_edge(name, members[(index + offset) % len(members)])

# A small number of explicit bridges ensures paths between communities.
community_keys = sorted(communities, key=str)
if len(community_keys) > 1:
    for index, community in enumerate(community_keys):
        next_community = community_keys[(index + 1) % len(community_keys)]
        social_graph.add_edge(
            network_rng.choice(communities[community]),
            network_rng.choice(communities[next_community]),
        )

# Heterogeneous target degrees and preferential attachment create a few hubs.
target_degree = {
    name: min(
        network_settings["degree_cap"],
        network_settings["degree_base"] + int(network_rng.paretovariate(network_settings["pareto_shape"])),
    )
    for name in agent_by_name
}
for source in agent_by_name:
    while social_graph.degree(source) < target_degree[source]:
        candidates = [
            name for name in agent_by_name
            if name != source and not social_graph.has_edge(source, name)
        ]
        if not candidates:
            break
        weights = [
            connection_affinity(source, candidate)
            * (social_graph.degree(candidate) + 1) ** 0.7
            for candidate in candidates
        ]
        social_graph.add_edge(
            source, network_rng.choices(candidates, weights=weights, k=1)[0]
        )

# Similar agents are more likely to have strong ties; bridges tend to be weak ties.
for left, right in social_graph.edges:
    affinity = connection_affinity(left, right)
    same_community = community_of(left) == community_of(right)
    strength = (
        "strong"
        if same_community and network_rng.random() < network_settings["strong_tie_probability"]
        else "weak"
    )
    social_graph.edges[left, right]["tie_strength"] = strength
    social_graph.edges[left, right]["weight"] = 2 if strength == "strong" else 1
    left_agent, right_agent = agent_by_name[left], agent_by_name[right]
    description = f"{strength.title()} social-network tie"
    left_agent.make_agent_accessible(right_agent, description)
    right_agent.make_agent_accessible(left_agent, description)

social_edges = {tuple(sorted(edge)) for edge in social_graph.edges}
social_edge_attributes = sorted(
    (min(left, right), max(left, right), data["tie_strength"])
    for left, right, data in social_graph.edges(data=True)
)
social_edges_df = nx.to_pandas_edgelist(social_graph)
neighbours = {person.name: [] for person in active_people}
neighbour_weights = {person.name: [] for person in active_people}
for name in neighbours:
    for peer in sorted(social_graph.neighbors(name)):
        neighbours[name].append(peer)
        neighbour_weights[name].append(social_graph.edges[name, peer]["weight"])

degree_values = [degree for _, degree in social_graph.degree()]
betweenness = nx.betweenness_centrality(social_graph)
nx.set_node_attributes(
    social_graph,
    {name: profile_by_name[name]["mother_tongue"] for name in agent_by_name},
    "mother_tongue",
)
network_diagnostics = {
    "average_degree": round(sum(degree_values) / len(degree_values), 2),
    "degree_range": [min(degree_values), max(degree_values)],
    "average_clustering": round(nx.average_clustering(social_graph), 3),
    "average_path_length": round(nx.average_shortest_path_length(social_graph), 2),
    "community_modularity": round(
        nx.community.modularity(social_graph, [set(members) for members in communities.values()]), 3
    ),
    "language_assortativity": round(
        nx.attribute_assortativity_coefficient(
            social_graph, "mother_tongue"
        ), 3
    ),
    "highest_betweenness_agent": max(betweenness, key=betweenness.get),
    "highest_betweenness": round(max(betweenness.values()), 3),
    "strong_ties": sum(
        data["tie_strength"] == "strong" for _, _, data in social_graph.edges(data=True)
    ),
}

world.broadcast_internal_goal(
    "Participate naturally in the policy discussion. Produce exactly one TALK action per round. "
    "Always speak in English. Give an independent reaction first; address a named participant only in later rounds. "
    f"Keep spoken responses conversational and within {response_word_limit} words."
)
policy_responses = []

signature_payload = {
    "network_preset": NETWORK_PRESET,
    "network_settings": network_settings,
    "policy_ids": selected_policy_ids,
    "policies": [policy_data["announcements"][key] for key in selected_policy_ids],
    "strategic_landscape": landscape_for_simulation,
    "agent_strategic_awareness": agent_strategic_awareness,
    "known_challenge_range": [minimum_known_challenges, maximum_known_challenges],
    "agents": [person._persona for person in active_people],
    "social_edges": sorted(social_edges),
    "social_edge_attributes": social_edge_attributes,
    "network_algorithm_version": 2,
    "network_diagnostics": network_diagnostics,
    "discussion_rounds": discussion_rounds,
    "response_word_limit": response_word_limit,
    "prompt_version": 8,
}
simulation_signature = hashlib.sha256(
    json.dumps(signature_payload, sort_keys=True).encode()
).hexdigest()
response_cache_dir = Path("cache/responses")
response_cache_dir.mkdir(parents=True, exist_ok=True)
response_cache_path = response_cache_dir / (
    f"{NETWORK_PRESET}-{simulation_signature[:16]}.json"
)

if USE_CACHE and response_cache_path.exists():
    response_cache = json.loads(response_cache_path.read_text(encoding="utf-8"))
else:
    response_cache = {}

total_agent_turns = len(selected_policy_ids) * discussion_rounds * len(active_people)
if response_cache.get("signature") == simulation_signature:
    policy_responses = response_cache["policy_responses"]
    progress = tqdm(
        total=total_agent_turns, initial=total_agent_turns,
        desc="Loaded cached simulation", unit="agent-turn", dynamic_ncols=True,
    )
    progress.close()
else:
    progress = tqdm(
        total=total_agent_turns, desc="Running simulation",
        unit="agent-turn", dynamic_ncols=True, smoothing=0.1,
    )
    for policy_id in selected_policy_ids:
        announcement = policy_data["announcements"][policy_id]
        policy_signature_payload = {
            key: value for key, value in signature_payload.items()
            if key not in {"policy_ids", "policies"}
        }
        policy_signature_payload.update({"policy_id": policy_id, "policy": announcement})
        policy_signature = hashlib.sha256(
            json.dumps(policy_signature_payload, sort_keys=True).encode()
        ).hexdigest()
        policy_cache_path = response_cache_dir / (
            f"{NETWORK_PRESET}-{policy_id}-{policy_signature[:16]}.json"
        )
        if USE_CACHE and policy_cache_path.exists():
            policy_cache = json.loads(policy_cache_path.read_text(encoding="utf-8"))
            if policy_cache.get("signature") == policy_signature:
                policy_responses.append(policy_cache["policy_response"])
                progress.update(discussion_rounds * len(active_people))
                progress.set_postfix(policy=policy_id, status="cached")
                continue
        stimulus = f"""
Government announcement: {announcement['title']}

{announcement['text']}

Discuss this announcement as a Canadian citizen. Focus on what genuinely matters to you;
you do not need to cover a fixed checklist of reaction, impact, support, and concerns.
"""
        world.broadcast(stimulus)
        actions_by_round = []
        reply_context_by_round = []
        previous_statements = {}
        for round_index in range(discussion_rounds):
            expected_targets = {}
            round_reply_context = {}
            for agent_index, agent in enumerate(active_people):
                if round_index == 0:
                    target = ""
                    round_instruction = (
                        "Before responding, privately consider every strategic challenge you were given. "
                        "If one materially affects your reaction, weave that connection concretely and naturally "
                        "into what you notice, prioritize, question, or worry about. If none is relevant, do not force one. "
                        "Give your own immediate, gut response to the policy. Do not name, address, "
                        "agree with, or reply to another participant. Use an empty TALK target. "
                    )
                else:
                    contacts = neighbours[agent.name]
                    target = network_rng.choices(
                        contacts, weights=neighbour_weights[agent.name], k=1
                    )[0]
                    target_statement = previous_statements[target]
                    round_reply_context[agent.name] = {
                        "target": target,
                        "statement": target_statement,
                    }
                    round_instruction = (
                        f"Respond directly to this statement made by {target} in the preceding round:\n"
                        f"\"{target_statement}\"\n"
                        "Privately consider whether any strategic challenge you know materially sharpens your reply. "
                        "When relevant, incorporate its substance naturally; otherwise do not force it. "
                        "Engage with one concrete idea from that statement by building on it, questioning it, "
                        "contrasting it, or disagreeing. Do not merely repeat your own earlier view. "
                    )
                expected_targets[agent.name] = target
                agent.internalize_goal(
                    round_instruction
                    + f"Your single TALK action must target exactly '{target}' and stay within {response_word_limit} words."
                )
            world.expected_targets = expected_targets
            raw_round_actions = world.run(
                    1,
                    return_actions=True,
                    randomize_agents_order=False,
                    parallelize=True,
                )[0]
            round_actions = {}
            current_statements = {}
            for agent_name, actions in raw_round_actions.items():
                talk_actions = [
                    action for action in actions
                    if action.get("action", action).get("type") == "TALK"
                ]
                matching_talk_actions = [
                    action for action in talk_actions
                    if (action.get("action", action).get("target") or "") == expected_targets[agent_name]
                ]
                if matching_talk_actions:
                    selected_action = matching_talk_actions[0]
                elif talk_actions:
                    # The content was generated from the intended person's quoted statement;
                    # repair only the malformed target field and deliver it once.
                    selected_action = copy.deepcopy(talk_actions[0])
                    selected_talk = selected_action.get("action", selected_action)
                    selected_talk["target"] = expected_targets[agent_name]
                    selected_action["target_was_corrected"] = True
                    if expected_targets[agent_name]:
                        world._handle_talk(
                            agent_by_name[agent_name], selected_talk.get("content", ""),
                            expected_targets[agent_name],
                        )
                else:
                    fallback_content = "I’m still considering that point and don’t have a clear response yet."
                    selected_action = {
                        "action": {
                            "type": "TALK", "content": fallback_content,
                            "target": expected_targets[agent_name],
                        },
                        "generation_status": "fallback_no_talk",
                    }
                    if expected_targets[agent_name]:
                        world._handle_talk(
                            agent_by_name[agent_name], fallback_content, expected_targets[agent_name]
                        )
                round_actions[agent_name] = [selected_action]
                selected_talk = selected_action.get("action", selected_action)
                current_statements[agent_name] = str(selected_talk.get("content", ""))
            actions_by_round.append(round_actions)
            reply_context_by_round.append(round_reply_context)
            previous_statements = current_statements
            progress.update(len(active_people))
            progress.set_postfix(policy=policy_id, round=round_index + 1)
        policy_result = {
                "policy_id": policy_id,
                "policy_title": announcement["title"],
                "policy_text": announcement["text"],
                "actions_by_round": actions_by_round,
                "reply_context_by_round": reply_context_by_round,
            }
        policy_responses.append(policy_result)
        policy_cache_path.write_text(
            json.dumps(
                {"signature": policy_signature, "policy_response": policy_result},
                indent=2,
            ),
            encoding="utf-8",
        )

    progress.close()
    response_cache_path.write_text(
        json.dumps(
            {
                "signature": simulation_signature,
                "policy_responses": policy_responses,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

Running simulation:   0%|          | 0/30 [00:00<?, ?agent-turn/s]

/opt/anaconda3/envs/tinytroupe/lib/python3.10/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CognitiveActionsModel(act...nd sustained support.')), input_type=CognitiveActionsModel])
  return self.__pydantic_serializer__.to_python(
/opt/anaconda3/envs/tinytroupe/lib/python3.10/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CognitiveActionsModel(act...bout reach and trust.')), input_type=CognitiveActionsModel])
  return self.__pydantic_serializer__.to_python(
/opt/anaconda3/envs/tinytroupe/lib/python3.10/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as

In [10]:
challenge_by_name = {
    challenge["name"]: challenge
    for challenge in strategic_landscape["challenges"]
}

def detect_invoked_challenges(agent_name, response):
    response_lower = response.lower()
    return [
        challenge_name
        for challenge_name in agent_strategic_awareness.get(agent_name, [])
        if any(
            signal.lower() in response_lower
            for signal in challenge_by_name[challenge_name].get("signals", [])
        )
    ]

response_rows = []
for result in policy_responses:
    for round_number, round_actions in enumerate(result["actions_by_round"], start=1):
        reply_context = result.get("reply_context_by_round", [{}] * len(result["actions_by_round"]))[round_number - 1]
        for agent_name, actions in round_actions.items():
            for action in actions:
                action_data = action.get("action", action)
                cognitive_state = action.get("cognitive_state", {})
                response_text = " ".join(
                    str(action_data.get("content", "")).split()[:response_word_limit]
                )
                response_rows.append(
                    {
                        "policy_id": result["policy_id"],
                        "policy_title": result["policy_title"],
                        "policy_text": result.get("policy_text", ""),
                        "round": round_number,
                        "agent_name": agent_name,
                        "action_type": action_data.get("type"),
                        "target": action_data.get("target"),
                        "reply_to_statement": reply_context.get(agent_name, {}).get("statement", ""),
                        "target_was_corrected": action.get("target_was_corrected", False),
                        "generation_status": action.get("generation_status", "generated"),
                        "known_challenges": agent_strategic_awareness.get(agent_name, []),
                        "invoked_challenges": detect_invoked_challenges(agent_name, response_text),
                        "response": response_text,
                        "original_word_count": len(
                            str(action_data.get("content", "")).split()
                        ),
                        "within_word_limit": len(
                            str(action_data.get("content", "")).split()
                        ) <= response_word_limit,
                        "attention": cognitive_state.get("attention"),
                        "emotions": cognitive_state.get("emotions"),
                    }
                )

responses_df = pd.DataFrame(response_rows)
responses_df

,policy_id,policy_title,policy_text,round,agent_name,action_type,target,reply_to_statement,target_was_corrected,generation_status,known_challenges,invoked_challenges,response,original_word_count,within_word_limit,attention,emotions
0,ai_literacy,Free introductory AI education,The Government of Canada announces plans for a...,1,Olivia Scott,TALK,,,False,generated,"[Fragmented and inaccessible data, Employment ...",[Employment disruption],This is a good step — free AI basics at librar...,33,True,"Accessibility for rural/remote communities, tr...",Cautiously optimistic but concerned about impl...
1,ai_literacy,Free introductory AI education,The Government of Canada announces plans for a...,1,Daniel Foster,TALK,,,False,generated,"[Privacy, discrimination, and online harms, Lo...","[Privacy, discrimination, and online harms, Em...",Glad to see free AI basics—it's needed. Hope p...,31,True,"Equitable access, rural and remote delivery, b...",Cautiously optimistic and pragmatic; mildly co...
2,ai_literacy,Free introductory AI education,The Government of Canada announces plans for a...,1,Claire Bennett,TALK,,,False,generated,"[Privacy, discrimination, and online harms, Ta...","[Privacy, discrimination, and online harms]",This is a good start — free AI basics in libra...,31,True,Focusing on ensuring the program teaches priva...,Cautiously optimistic but concerned about impl...
3,ai_literacy,Free introductory AI education,The Government of Canada announces plans for a...,1,Liam Carter,TALK,,,False,generated,"[Low public trust and AI literacy, Privacy, di...",[],Good idea. People need simple AI basics. Libra...,32,True,"Reaching remote, older, and low-income people;...",Cautiously hopeful but concerned about reach a...
4,ai_literacy,Free introductory AI education,The Government of Canada announces plans for a...,1,Sophie Hayes,TALK,,,False,generated,"[Talent retention and commercialization, Fragm...","[Talent retention and commercialization, Busin...",I welcome free AI basics — a smart step for in...,34,True,"Prioritizing inclusion and safety benefits, an...","Encouraged but cautious; supportive of access,..."
5,ai_literacy,Free introductory AI education,The Government of Canada announces plans for a...,1,Lucien Dupuis,TALK,,,False,generated,"[Unequal access and disproportionate harms, Em...",[],Good idea — free AI classes help people. I wor...,33,True,"Rural internet/power access, availability of F...",Cautiously hopeful but worried about access an...
6,ai_literacy,Free introductory AI education,The Government of Canada announces plans for a...,1,Arjun Patel,TALK,,,False,generated,"[Unequal access and disproportionate harms, Ta...","[Unequal access and disproportionate harms, Pr...",This is a good start. Free AI classes could he...,34,True,"Equitable reach (rural, Indigenous), trainer q...",Cautious optimism mixed with concern about equ...
7,ai_literacy,Free introductory AI education,The Government of Canada announces plans for a...,1,Priya Desai,TALK,,,False,generated,"[Unequal access and disproportionate harms, Ta...",[],"Good idea — free AI classes could help, but I'...",31,True,"Rural internet, library capacity, trained teac...",Cautiously hopeful but worried about access an...
8,ai_literacy,Free introductory AI education,The Government of Canada announces plans for a...,1,Evan Clarke,TALK,,,False,generated,"[Talent retention and commercialization, Low p...","[Employment disruption, Privacy, discriminatio...",Good move — free AI basics in libraries will h...,36,True,"Focusing on delivery to rural communities, pra...",Supportive but cautious — optimistic about acc...
9,ai_literacy,Free introductory AI education,The Government of Canada announces plans for a...,1,Ethan Wallace,TALK,,,False,generated,"[Unequal access and disproportionate harms, Pr...","[Unequal access and disproportionate harms, Pr...",I like the idea — free AI literacy can help. M...,33,True,"Whether the program will reach remote, Indigen...",Cautiously supportive but 

In [11]:
outputs_path = Path("outputs/scalable_simulation") / NETWORK_PRESET
outputs_path.mkdir(parents=True, exist_ok=True)
responses_df.to_excel(outputs_path / "ai_literacy.xlsx", index=False)

# Animated social-network conversation

The notebook displays a rotating group conversation. Hover over any demographic-aware figure for their profile; the active speech appears above the group.

In [12]:
import html
from IPython.display import HTML, display

talk_events = responses_df.loc[
    responses_df["action_type"].eq("TALK"),
    ["policy_id", "policy_title", "policy_text", "round", "agent_name", "target", "reply_to_statement", "known_challenges", "invoked_challenges", "response", "attention"],
] .fillna("").to_dict(orient="records")

network_positions = nx.spring_layout(
    social_graph, seed=RANDOM_SEED, weight="weight", iterations=100
)
payload = {
    "events": talk_events,
    "profiles": {
    person.name: profile for person, profile in zip(active_people, active_profiles)
    },
    "names": [person.name for person in active_people],
    "network_preset": NETWORK_PRESET,
    "network_preset_label": network_settings["label"],
    "available_presets": [
        {
            "id": preset_id,
            "label": settings["label"],
            "url": f"../{preset_id}/conversation_animation.html",
        }
        for preset_id, settings in NETWORK_PRESETS.items()
        if preset_id == NETWORK_PRESET
        or (Path("outputs/scalable_simulation") / preset_id / "conversation_animation.html").exists()
    ],
    "strategic_awareness": agent_strategic_awareness,
    "social_edges": sorted(social_edges),
    "network_diagnostics": network_diagnostics,
    "network_nodes": [
        {
            "name": name,
            "x": round(float(network_positions[name][0]), 5),
            "y": round(float(network_positions[name][1]), 5),
            "community": " / ".join(map(str, community_of(name))),
            "degree": social_graph.degree(name),
            "betweenness": round(float(betweenness[name]), 5),
        }
        for name in social_graph.nodes
    ],
    "network_edges": [
        {"source": left, "target": right, "strength": data["tie_strength"]}
        for left, right, data in social_graph.edges(data=True)
    ],
    "network": {
        "participants": len(active_people),
        "relationships": len(social_edges),
        "average_contacts": round(sum(map(len, neighbours.values())) / len(neighbours), 1),
        "minimum_contacts": min(map(len, neighbours.values())),
        "maximum_contacts": max(map(len, neighbours.values())),
        "density": round(2 * len(social_edges) / (len(active_people) * (len(active_people) - 1)), 3),
        "structure": "Homophilic demographic communities, heterogeneous degrees, strong ties, hubs, and cross-community bridges",
    },
}
payload_json = json.dumps(payload).replace("</", "<\\/")
document = r'''<!doctype html><meta charset="utf-8">
<style>
:root{--talk:#00cd00;--think:#008000;--conversation:#00ffff;--reach:#800080;--done:#d1d1d1}*{box-sizing:border-box}body{margin:0;background:linear-gradient(145deg,#fffaf1,#f5fbff);color:#352f2a;font:14px system-ui,sans-serif}.stage{max-width:1100px;margin:auto;padding:20px;text-align:center}.eyebrow{color:#766b62;font-size:11px;font-weight:700;letter-spacing:.13em;text-transform:uppercase}.network{display:flex;gap:8px;justify-content:center;flex-wrap:wrap;margin:10px auto}.stat{min-width:110px;padding:7px 11px;border:1px solid #ddd4ca;border-radius:10px;background:#ffffffc9}.stat b{display:block;font-size:17px;color:#5f276d}.network-note{color:#766b62;font-size:11px;margin-top:-3px}.policy{max-width:820px;margin:10px auto;padding:12px 16px;border-left:5px solid var(--reach);border-radius:8px;background:#fff;text-align:left;line-height:1.4;box-shadow:0 3px 12px #0001}.policy summary{cursor:pointer;font-weight:750;color:#5f276d}.policy div{margin-top:8px;white-space:pre-wrap}.reply{max-width:680px;margin:10px auto -10px;padding:10px 15px;border-radius:14px;background:#e8ffff;border:2px solid var(--conversation);text-align:left;font-size:12px}.legend{display:flex;gap:14px;justify-content:center;flex-wrap:wrap;margin:9px 0;font-size:12px}.dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:4px}.progress{height:5px;max-width:760px;margin:10px auto;background:#e5e7eb;border-radius:9px;overflow:hidden}.progress i{display:block;height:100%;width:0;background:linear-gradient(90deg,var(--talk),var(--conversation));transition:width .35s}.controls{display:flex;gap:8px;align-items:center;justify-content:center}.controls input{width:min(600px,60vw)}button{padding:7px 13px;border:1px solid #c9c2b9;border-radius:8px;background:white;cursor:pointer}.title{font-size:22px;font-weight:750}.meta{color:#766b62;margin:4px}.speech{min-height:78px;max-width:720px;margin:20px auto 12px;padding:17px 20px;border-radius:22px;background:#f3fff3;border:3px solid var(--talk);text-align:left;font-size:15px;line-height:1.45;box-shadow:0 6px 20px #0002}.speech.pulse{animation:pop .35s ease-out}.people{display:flex;justify-content:space-around;align-items:end;min-height:245px}.person{position:relative;width:145px;padding:9px;border:3px solid transparent;border-radius:18px;transition:transform .25s,opacity .25s}.person:not(.speaker):not(.target){opacity:.68}.person.speaker{background:#eaffea;border-color:var(--talk);transform:translateY(-9px)}.person.target{background:#e8ffff;border-color:var(--conversation)}.icon{display:block;font-size:58px}.speaker .icon{animation:bob 1s ease-in-out infinite alternate}.role{min-height:16px;font-size:9px;font-weight:800;letter-spacing:.08em}.speaker .role{color:#007a00}.target .role{color:#007f87}.name{font-weight:700;font-size:12px}.tip{display:none;position:absolute;z-index:10;bottom:100%;left:50%;transform:translateX(-50%);width:245px;padding:10px;background:#26211e;color:white;border-radius:9px;text-align:left;font-size:12px;box-shadow:0 5px 18px #0005}.person:hover{opacity:1}.person:hover .tip{display:block}.hint{margin:10px;color:#857970;font-size:11px}@keyframes bob{to{transform:translateY(-5px) rotate(2deg)}}@keyframes pop{0%{transform:scale(.97);opacity:.4}100%{transform:scale(1);opacity:1}}@media(max-width:720px){.people{overflow-x:auto;justify-content:flex-start;gap:8px}.person{min-width:120px}.speech{font-size:13px}}
.invoked{max-width:720px;margin:12px auto -10px;color:#5f276d;font-size:12px}.invoked b{margin-right:6px}.challenge-chip{display:inline-block;margin:3px;padding:4px 8px;border-radius:12px;background:#f4e8f7;border:1px solid var(--reach)}
.sample-control{display:flex;gap:10px;align-items:center;justify-content:center;margin:12px auto;color:#5f276d;font-weight:700}.sample-control select{padding:6px 10px;border:1px solid #c9c2b9;border-radius:8px;background:white;color:#352f2a;font-weight:700}
.diagnostics{max-width:900px;margin:7px auto 12px;color:#766b62;font-size:11px;line-height:1.5}.diagnostics b{color:#5f276d}
</style><div class="stage"><div class="eyebrow">TinyTroupe policy simulation</div><div class="title" id="title"></div><div class="network" id="network"></div><div class="network-note" id="network-note"></div><details class="policy" open><summary>Policy announcement</summary><div id="policy"></div></details><div class="meta" id="meta"></div><div class="progress"><i id="bar"></i></div><div class="legend"><span><i class="dot" style="background:var(--talk)"></i>TALK</span><span><i class="dot" style="background:var(--think)"></i>THINK</span><span><i class="dot" style="background:var(--conversation)"></i>CONVERSATION</span><span><i class="dot" style="background:var(--reach)"></i>REACH_OUT</span><span><i class="dot" style="background:var(--done)"></i>DONE</span></div><div class="reply" id="reply"></div><div class="speech" id="speech"></div><div class="people" id="people"></div><div class="controls"><button id="prev" title="Previous message">◀</button><button id="play">▶ Play</button><button id="next" title="Next message">▶</button><input id="timeline" type="range" min="0" value="0" aria-label="Conversation timeline"><span id="count"></span></div><div class="hint">Hover over a person for demographics · use ← and → to navigate · green speaks, cyan listens</div></div>
<script>
const data=__PAYLOAD__,allEvents=data.events,recent=[];let sampleSize=data.names.length,events=[...allEvents];let index=0,timer=null;
const esc=s=>String(s??'').replace(/[&<>\"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','\"':'&quot;'}[c]));
const invokedBox=document.createElement('div');invokedBox.id='invoked';invokedBox.className='invoked';document.getElementById('speech').before(invokedBox);
function renderNetwork(){const chosen=new Set(data.names.slice(0,sampleSize)),edges=data.social_edges.filter(([a,b])=>chosen.has(a)&&chosen.has(b)),degrees=Object.fromEntries([...chosen].map(n=>[n,0]));edges.forEach(([a,b])=>{degrees[a]++;degrees[b]++});const values=Object.values(degrees),net={participants:chosen.size,relationships:edges.length,average_contacts:chosen.size?(2*edges.length/chosen.size).toFixed(1):0,minimum_contacts:values.length?Math.min(...values):0,maximum_contacts:values.length?Math.max(...values):0,density:chosen.size>1?(2*edges.length/(chosen.size*(chosen.size-1))).toFixed(3):0};document.getElementById('network').innerHTML=[['Participants',net.participants],['Relationships',net.relationships],['Average contacts',net.average_contacts],['Contact range',net.minimum_contacts+'–'+net.maximum_contacts],['Density',net.density]].map(([label,value])=>'<div class="stat"><b>'+esc(value)+'</b>'+esc(label)+'</div>').join('');document.getElementById('network-note').textContent='Selected induced subnetwork · demographic communities, varied degrees, strong ties, hubs, and bridges'}
const cohortSizes=[10,30,75,150,300].filter(n=>n<=data.names.length);if(!cohortSizes.includes(data.names.length))cohortSizes.push(data.names.length);const sampleControl=document.createElement('div');sampleControl.className='sample-control';sampleControl.innerHTML='<label for="sample-size">Displayed nested view (not a new simulation)</label><select id="sample-size">'+cohortSizes.map(n=>'<option value="'+n+'"'+(n===sampleSize?' selected':'')+'>'+n+' agents</option>').join('')+'</select>';document.getElementById('network').before(sampleControl);document.getElementById('sample-size').onchange=e=>{sampleSize=+e.target.value;const chosen=new Set(data.names.slice(0,sampleSize));events=allEvents.filter(x=>chosen.has(x.agent_name)&&(!x.target||chosen.has(x.target)));recent.length=0;index=0;const line=document.getElementById('timeline');line.max=Math.max(0,events.length-1);line.value=0;renderNetwork();if(events.length)render();else document.getElementById('speech').textContent='No conversations are available for this cohort.'};renderNetwork();
const d=data.network_diagnostics,diagnostics=document.createElement('div');diagnostics.className='diagnostics';diagnostics.innerHTML='<b>Active preset: '+esc(data.network_preset_label)+' ('+esc(data.network_preset)+').</b> Clustering '+esc(d.average_clustering)+' · average path '+esc(d.average_path_length)+' · community modularity '+esc(d.community_modularity)+' · language assortativity '+esc(d.language_assortativity)+' · strong ties '+esc(d.strong_ties)+' · highest-betweenness bridge: '+esc(d.highest_betweenness_agent)+' ('+esc(d.highest_betweenness)+')';document.getElementById('network-note').after(diagnostics);
function profile(name){const p=data.profiles[name]||{},known=data.strategic_awareness[name]||[];return '<b>'+esc(name)+'</b><br>'+Object.entries(p).filter(([k])=>k!=='profile_id').map(([k,v])=>esc(k.replaceAll('_',' ').replace(/\\b\\w/g,c=>c.toUpperCase()))+': '+esc(v)).join('<br>')+'<hr><b>Known strategic challenges</b><br>'+known.map(esc).join('<br>')}
function avatar(name){const p=data.profiles[name]||{},race=p.race_ethnicity||'',gender=String(p.gender||'').toLowerCase();const tone=race==='Black'?'🏿':['South Asian','Indigenous','Filipino','Arab','Latin American'].includes(race)?'🏽':['White','Chinese'].includes(race)?'🏻':'';const base=gender.includes('woman')||gender.includes('female')?'👩':gender.includes('man')||gender.includes('male')?'👨':'🧑';return base+tone}
function render(){const e=events[index],candidates=[e.agent_name,e.target,...recent.slice().reverse(),...data.names],visible=[];for(const n of candidates){if(n&&!visible.includes(n))visible.push(n);if(visible.length===Math.min(6,data.names.length))break}recent.push(e.agent_name);if(recent.length>8)recent.shift();document.getElementById('title').textContent=e.policy_title;document.getElementById('policy').textContent=e.policy_text;document.getElementById('meta').textContent='Round '+e.round+' · '+(e.target?e.agent_name+' → '+e.target:'independent reaction by '+e.agent_name)+' · message '+(index+1)+'/'+events.length;const reply=document.getElementById('reply');reply.style.display=e.reply_to_statement?'block':'none';reply.innerHTML=e.reply_to_statement?'↩ <b>Responding to '+esc(e.target)+':</b> “'+esc(e.reply_to_statement)+'”':'';const speech=document.getElementById('speech');speech.innerHTML='💬 <b>'+esc(e.agent_name)+'</b><br>'+esc(e.response);speech.classList.remove('pulse');void speech.offsetWidth;speech.classList.add('pulse');document.getElementById('people').innerHTML=visible.map(n=>'<div class="person '+(n===e.agent_name?'speaker':n===e.target?'target':'')+'"><span class="icon">'+avatar(n)+'</span><div class="role">'+(n===e.agent_name?'SPEAKING':n===e.target?'LISTENING':'IN THE GROUP')+'</div><div class="name">'+esc(n)+'</div><div class="tip">'+profile(n)+'</div></div>').join('');document.getElementById('bar').style.width=((index+1)/events.length*100)+'%';timeline.value=index;count.textContent=(index+1)+'/'+events.length}
const baseRender=render;render=function(){baseRender();const invoked=events[index].invoked_challenges||[];invokedBox.innerHTML=invoked.length?'<b>Possible strategic context invoked:</b>'+invoked.map(x=>'<span class="challenge-chip">'+esc(x)+'</span>').join(''):'<span>No clear strategic-context signal in this response</span>'}
const timeline=document.getElementById('timeline'),count=document.getElementById('count');timeline.max=Math.max(0,events.length-1);timeline.oninput=()=>{index=+timeline.value;render()};prev.onclick=()=>{index=(index-1+events.length)%events.length;render()};next.onclick=()=>{index=(index+1)%events.length;render()};play.onclick=()=>{if(timer){clearInterval(timer);timer=null;play.textContent='▶ Play'}else{timer=setInterval(()=>{index=(index+1)%events.length;render()},2200);play.textContent='❚❚ Pause'}};document.onkeydown=e=>{if(e.key==='ArrowLeft')prev.click();if(e.key==='ArrowRight')next.click();if(e.key===' ')play.click()};if(events.length)render();else document.body.textContent='No TALK events are available to animate.';
</script>'''.replace("__PAYLOAD__", payload_json)
template_path = Path("templates/conversation_visualization.html")
document = template_path.read_text(encoding="utf-8").replace(
    "__PAYLOAD__", payload_json
)
generated_presets = [
    {
        "id": preset_id,
        "label": settings["label"],
        "url": f"../{preset_id}/conversation_animation.html",
    }
    for preset_id, settings in NETWORK_PRESETS.items()
    if preset_id == NETWORK_PRESET
    or (outputs_path.parent / preset_id / "conversation_animation.html").exists()
]
(outputs_path.parent / "available_presets.js").write_text(
    "window.AVAILABLE_PRESETS = " + json.dumps(generated_presets) + ";\n",
    encoding="utf-8",
)
standalone_path = outputs_path / "conversation_animation.html"
standalone_path.write_text(document, encoding="utf-8")
display(HTML(
    f'<p><a href="{standalone_path.as_posix()}" target="_blank">Open standalone conversation animation</a></p>'
    f'<p><code>{standalone_path.resolve()}</code></p>'
))
